
# Round2 vs Round4 실제 대피안내도 비교

같은 실제 대피도 이미지에 대해 Round2와 Round4를 동일 조건으로 추론하고,
검출 수/신뢰도/비교 이미지를 자동 저장합니다.

클래스 순서는 절대 변경하지 않습니다.

0 exit  
1 stair  
2 elevator  
3 extinguisher  
4 hydrant  
5 you_are_here  
6 door  
7 room


In [ ]:

# Cell 1 — 설치 및 Drive 연결
!pip -q install ultralytics pandas

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import shutil
import pandas as pd
import numpy as np
import cv2
from ultralytics import YOLO

CLASS_NAMES = [
    "exit",
    "stair",
    "elevator",
    "extinguisher",
    "hydrant",
    "you_are_here",
    "door",
    "room",
]

assert CLASS_NAMES == [
    "exit","stair","elevator","extinguisher",
    "hydrant","you_are_here","door","room"
]

print("✅ 준비 완료")


In [ ]:
from pathlib import Path
import os

ROOT = Path("/content/drive/MyDrive")

IMAGE_EXTS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp",
    ".tif", ".tiff"
}

candidates = []

for root, dirs, files in os.walk(ROOT):
    imgs = [
        f for f in files
        if Path(f).suffix.lower() in IMAGE_EXTS
    ]

    if len(imgs) >= 3:
        candidates.append((len(imgs), Path(root)))

candidates.sort(reverse=True, key=lambda x: x[0])

print("실제 이미지 폴더 후보:")
for n, p in candidates[:50]:
    print(f"{n:4d} images | {p}")

In [ ]:

# Cell 2 — 경로 설정

ROUND2_MODEL = Path(
    "/content/drive/MyDrive/evacuation_checkpoints/"
    "round2/evac_round2_stage_a/weights/best.pt"
)

ROUND4_MODEL = Path(
    "/content/drive/MyDrive/evacuation_yolo/"
    "round4_clean/guarded_best.pt"
)

# 실제 비교할 대피안내도 원본 폴더
REAL_IMAGE_DIR = Path(
    "/content/drive/MyDrive/02/ml/real_data/photos_all"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/evacuation_yolo/"
    "round4_clean/round2_vs_round4_compare"
)

IMG_SIZE = 768
CONF = 0.25
IOU = 0.50
DEVICE = 0

print("ROUND2_MODEL :", ROUND2_MODEL)
print("ROUND4_MODEL :", ROUND4_MODEL)
print("REAL_IMAGE_DIR:", REAL_IMAGE_DIR)
print("OUTPUT_DIR   :", OUTPUT_DIR)

assert ROUND2_MODEL.exists(), f"Round2 모델 없음: {ROUND2_MODEL}"
assert ROUND4_MODEL.exists(), f"Round4 모델 없음: {ROUND4_MODEL}"
assert REAL_IMAGE_DIR.exists(), f"실제 이미지 폴더 없음: {REAL_IMAGE_DIR}"

print("\n✅ 경로 확인 완료")


In [ ]:

# Cell 3 — 실제 이미지 확인

IMAGE_EXTS = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}

images = sorted([
    p for p in REAL_IMAGE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
])

print("실제 비교 이미지:", len(images))

for p in images[:20]:
    print(" ", p)

if len(images) == 0:
    raise RuntimeError(
        "비교할 실제 이미지가 없습니다.\n"
        "Cell 2의 REAL_IMAGE_DIR를 실제 대피안내도 원본 폴더로 바꾸세요."
    )


In [ ]:

# Cell 4 — Round2 / Round4 동시 추론

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

r2_dir = OUTPUT_DIR / "round2"
r4_dir = OUTPUT_DIR / "round4"
side_dir = OUTPUT_DIR / "side_by_side"

r2_dir.mkdir(parents=True, exist_ok=True)
r4_dir.mkdir(parents=True, exist_ok=True)
side_dir.mkdir(parents=True, exist_ok=True)

model_r2 = YOLO(str(ROUND2_MODEL))
model_r4 = YOLO(str(ROUND4_MODEL))

def normalize_names(names):
    if isinstance(names, dict):
        return [names[i] for i in range(len(names))]
    return list(names)

assert normalize_names(model_r2.names) == CLASS_NAMES, "Round2 클래스 순서 불일치"
assert normalize_names(model_r4.names) == CLASS_NAMES, "Round4 클래스 순서 불일치"

rows = []

for idx, img_path in enumerate(images, start=1):
    print(f"[{idx}/{len(images)}] {img_path.name}")

    r2 = model_r2.predict(
        source=str(img_path),
        imgsz=IMG_SIZE,
        conf=CONF,
        iou=IOU,
        device=DEVICE,
        verbose=False,
    )[0]

    r4 = model_r4.predict(
        source=str(img_path),
        imgsz=IMG_SIZE,
        conf=CONF,
        iou=IOU,
        device=DEVICE,
        verbose=False,
    )[0]

    vis2 = r2.plot()
    vis4 = r4.plot()

    cv2.imwrite(str(r2_dir / f"{img_path.stem}_round2.jpg"), vis2)
    cv2.imwrite(str(r4_dir / f"{img_path.stem}_round4.jpg"), vis4)

    h = max(vis2.shape[0], vis4.shape[0])

    def pad_to_h(im, target_h):
        if im.shape[0] == target_h:
            return im
        pad = target_h - im.shape[0]
        return cv2.copyMakeBorder(
            im, 0, pad, 0, 0,
            cv2.BORDER_CONSTANT,
            value=(255,255,255)
        )

    vis2p = pad_to_h(vis2, h)
    vis4p = pad_to_h(vis4, h)
    side = np.hstack([vis2p, vis4p])

    cv2.putText(side, "Round2", (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,0), 2, cv2.LINE_AA)
    cv2.putText(side, "Round4", (vis2p.shape[1] + 20, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,0), 2, cv2.LINE_AA)

    cv2.imwrite(str(side_dir / f"{img_path.stem}_compare.jpg"), side)

    def collect(result):
        stats = {}
        if result.boxes is None or len(result.boxes) == 0:
            for cid in range(len(CLASS_NAMES)):
                stats[cid] = {"count":0, "mean_conf":0.0, "max_conf":0.0}
            return stats

        cls = result.boxes.cls.detach().cpu().numpy().astype(int)
        conf = result.boxes.conf.detach().cpu().numpy()

        for cid in range(len(CLASS_NAMES)):
            mask = (cls == cid)
            vals = conf[mask]
            stats[cid] = {
                "count": int(mask.sum()),
                "mean_conf": float(vals.mean()) if len(vals) else 0.0,
                "max_conf": float(vals.max()) if len(vals) else 0.0,
            }
        return stats

    s2 = collect(r2)
    s4 = collect(r4)

    for cid, cname in enumerate(CLASS_NAMES):
        rows.append({
            "image": img_path.name,
            "class_id": cid,
            "class_name": cname,
            "round2_count": s2[cid]["count"],
            "round4_count": s4[cid]["count"],
            "count_delta": s4[cid]["count"] - s2[cid]["count"],
            "round2_mean_conf": s2[cid]["mean_conf"],
            "round4_mean_conf": s4[cid]["mean_conf"],
            "round2_max_conf": s2[cid]["max_conf"],
            "round4_max_conf": s4[cid]["max_conf"],
        })

df = pd.DataFrame(rows)
df.to_csv(
    OUTPUT_DIR / "image_class_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ 추론 완료")


In [ ]:

# Cell 5 — 클래스별 비교 요약

summary = (
    df.groupby(["class_id","class_name"], as_index=False)
      .agg(
          round2_total=("round2_count","sum"),
          round4_total=("round4_count","sum"),
          count_delta=("count_delta","sum"),
          round2_avg_conf=("round2_mean_conf","mean"),
          round4_avg_conf=("round4_mean_conf","mean"),
      )
)

summary["count_ratio_r4_vs_r2"] = (
    summary["round4_total"] /
    summary["round2_total"].replace(0, np.nan)
)

summary.to_csv(
    OUTPUT_DIR / "class_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("===== 전체 클래스 비교 =====")
display(summary)

print("\n===== 핵심 클래스 =====")
core = summary[summary["class_id"].isin([0,1,5])].copy()
display(core)

print("\n주의:")
print("- GT 없는 실제 이미지에서는 detection 개수 증가가 곧 정확도 향상을 의미하지 않습니다.")
print("- 특히 stair 증가가 실제 stair 검출인지 false positive인지 side_by_side 이미지를 확인해야 합니다.")


In [ ]:

# Cell 6 — 핵심 클래스 변화가 큰 이미지 자동 추출

core_df = df[df["class_id"].isin([0,1,5])].copy()

interesting = (
    core_df.assign(abs_delta=core_df["count_delta"].abs())
           .sort_values(
               ["abs_delta","class_id","image"],
               ascending=[False, True, True]
           )
)

interesting.to_csv(
    OUTPUT_DIR / "core_interesting_cases.csv",
    index=False,
    encoding="utf-8-sig"
)

print("===== 변화가 큰 핵심 클래스 사례 =====")
display(interesting.head(50))

print("\n비교 이미지 폴더:")
print(OUTPUT_DIR / "side_by_side")


In [ ]:

# Cell 7 — 최종 간단 요약

core = summary[summary["class_id"].isin([0,1,5])].set_index("class_name")

print("===================================")
print("Round2 vs Round4 REAL comparison")
print("===================================")

for cname in ["exit","stair","you_are_here"]:
    r = core.loc[cname]
    print(
        f"{cname:12s} | "
        f"Round2 {int(r['round2_total']):4d} -> "
        f"Round4 {int(r['round4_total']):4d} "
        f"(delta {int(r['count_delta']):+d})"
    )

print("\n결과 위치:")
print(OUTPUT_DIR)
print("\n중요 파일:")
print(" - image_class_comparison.csv")
print(" - class_summary.csv")
print(" - core_interesting_cases.csv")
print(" - side_by_side/")
print("\n✅ 비교 완료")
